# Lab 5: EDA with SQL

**Goal:** Load the launch data into a local SQLite database and answer analytical questions with SQL: which launch sites are used, how payload mass relates to outcomes, success rates by year, and rankings of landing outcome types. We use SQLite here (built into Python, no server setup) so this runs anywhere, including Colab.

In [1]:
import pandas as pd
import sqlite3
import os

## 0. Load the dataset and create the SQL table

In [2]:
if os.path.exists('dataset_part_2.csv'):
    df = pd.read_csv('dataset_part_2.csv')
else:
    print('dataset_part_2.csv not found locally, loading from IBM dataset repository...')
    fallback_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
    df = pd.read_csv(fallback_url)

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()

conn = sqlite3.connect(':memory:')
df.to_sql('SPACEXTABLE', conn, index=False, if_exists='replace')

print('Table loaded with', df.shape[0], 'rows')
pd.read_sql("SELECT * FROM SPACEXTABLE LIMIT 5", conn)

dataset_part_2.csv not found locally, loading from IBM dataset repository...
Table loaded with 90 rows


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class,Year,Month
0,1,2010-06-04 00:00:00,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0,2010,June
1,2,2012-05-22 00:00:00,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0,2012,May
2,3,2013-03-01 00:00:00,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0,2013,March
3,4,2013-09-29 00:00:00,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,0,0,0,None,1.0,0,B1003,-120.610829,34.632093,0,2013,September
4,5,2013-12-03 00:00:00,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0,2013,December


## 1. Distinct launch sites used in the missions

In [3]:
pd.read_sql("SELECT DISTINCT LaunchSite FROM SPACEXTABLE", conn)

,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


## 2. Total and average payload mass by launch site

In [4]:
pd.read_sql("""
SELECT LaunchSite,
       COUNT(*) AS TotalLaunches,
       ROUND(SUM(PayloadMass), 1) AS TotalPayloadMass,
       ROUND(AVG(PayloadMass), 1) AS AvgPayloadMass
FROM SPACEXTABLE
GROUP BY LaunchSite
ORDER BY TotalLaunches DESC
""", conn)

,LaunchSite,TotalLaunches,TotalPayloadMass,AvgPayloadMass
0,CCAFS SLC 40,55,305151.4,5548.2
1,KSC LC 39A,22,167341.9,7606.5
2,VAFB SLC 4E,13,76953.0,5919.5


## 3. Overall and per-site landing success rate

In [5]:
print("Overall success rate:")
display(pd.read_sql("SELECT ROUND(AVG(Class) * 100, 1) AS SuccessRatePct FROM SPACEXTABLE", conn))

print("Success rate by launch site:")
display(pd.read_sql("""
SELECT LaunchSite,
       COUNT(*) AS TotalLaunches,
       SUM(Class) AS SuccessfulLandings,
       ROUND(AVG(Class) * 100, 1) AS SuccessRatePct
FROM SPACEXTABLE
GROUP BY LaunchSite
ORDER BY SuccessRatePct DESC
""", conn))

Overall success rate:


,SuccessRatePct
0,66.7


Success rate by launch site:


,LaunchSite,TotalLaunches,SuccessfulLandings,SuccessRatePct
0,KSC LC 39A,22,17,77.3
1,VAFB SLC 4E,13,10,76.9
2,CCAFS SLC 40,55,33,60.0


## 4. Ranking landing outcome types by frequency

In [6]:
pd.read_sql("""
SELECT Outcome, COUNT(*) AS Frequency
FROM SPACEXTABLE
GROUP BY Outcome
ORDER BY Frequency DESC
""", conn)

,Outcome,Frequency
0,True ASDS,41
1,None None,19
2,True RTLS,14
3,False ASDS,6
4,True Ocean,5
5,None ASDS,2
6,False Ocean,2
7,False RTLS,1


## 5. Booster with the maximum payload mass carried

In [7]:
pd.read_sql("""
SELECT BoosterVersion, Serial, PayloadMass, LaunchSite, Date
FROM SPACEXTABLE
WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTABLE)
""", conn)

,BoosterVersion,Serial,PayloadMass,LaunchSite,Date
0,Falcon 9,B1048,15600.0,CCAFS SLC 40,2019-11-11 00:00:00
1,Falcon 9,B1051,15600.0,CCAFS SLC 40,2020-01-29 00:00:00
2,Falcon 9,B1048,15600.0,KSC LC 39A,2020-03-18 00:00:00


## 6. Date of the first successful landing

In [8]:
pd.read_sql("""
SELECT MIN(Date) AS FirstSuccessfulLanding
FROM SPACEXTABLE
WHERE Class = 1
""", conn)

,FirstSuccessfulLanding
0,2014-04-18 00:00:00


## 7. Success rate trend by year (time analysis)

In [9]:
pd.read_sql("""
SELECT Year,
       COUNT(*) AS TotalLaunches,
       SUM(Class) AS SuccessfulLandings,
       ROUND(AVG(Class) * 100, 1) AS SuccessRatePct
FROM SPACEXTABLE
GROUP BY Year
ORDER BY Year
""", conn)

,Year,TotalLaunches,SuccessfulLandings,SuccessRatePct
0,2010,1,0,0.0
1,2012,1,0,0.0
2,2013,3,0,0.0
3,2014,6,2,33.3
4,2015,6,2,33.3
5,2016,8,5,62.5
6,2017,18,15,83.3
7,2018,18,11,61.1
8,2019,10,9,90.0
9,2020,19,16,84.2


## 8. Boosters that failed to land in months with a specific pattern (example: 2015 failures by month)

In [10]:
pd.read_sql("""
SELECT Month, Date, BoosterVersion, LaunchSite, Outcome
FROM SPACEXTABLE
WHERE Year = 2015 AND Class = 0
""", conn)

,Month,Date,BoosterVersion,LaunchSite,Outcome
0,January,2015-01-10 00:00:00,Falcon 9,CCAFS SLC 40,False ASDS
1,April,2015-04-14 00:00:00,Falcon 9,CCAFS SLC 40,False ASDS
2,April,2015-04-27 00:00:00,Falcon 9,CCAFS SLC 40,None None
3,June,2015-06-28 00:00:00,Falcon 9,CCAFS SLC 40,None ASDS


## 9. Payload mass ranking within a specific date range

In [11]:
pd.read_sql("""
SELECT Outcome, COUNT(*) AS Frequency
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Outcome
ORDER BY Frequency DESC
""", conn)

,Outcome,Frequency
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2
